# exp_4 — SetSeq + touch features + auxiliary hidden-ball head (pre-registered in `PREREG.md`)

**Question (H1).** exp_3 showed that team assignment, not dead/alive, is the bottleneck: short possessions are
missed even with the ball visible, and half the misses sit in ball gaps. Does SetSeq with the 4 sub-step touch
features **and** a training-only auxiliary head that predicts the hidden clean ball and the team nearest to it
(BallRadar-style) beat exp_2's 3-seed ensemble (`exp3_ref_ens3`) on provider labels, while staying non-inferior
on `L_ctrl`? 3 seeds per arm, provider training labels, nested leave-one-match-out on 7 degraded IDSSE matches.

| component | exp_4 | closest prior work |
|---|---|---|
| auxiliary supervision | clean ball (x, y) + nearest team, training matches only, unused at inference | none in the team; BallRadar (Kim et al., KDD 2023) predicts the ball from players |
| touch features | exp_3's 4 features, now 3 seeds on provider labels | exp_3 S3: 1 seed, `L_ctrl` labels |
| overfitting controls | frozen prereg, fixed `TrainConfig()`, NLL generalisation gap per fit, seed honesty | exp_1–exp_3: prereg + nested selection |


In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp4_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_4')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp4_common as C
import exp1_common as C1
if not C.PROTOCOL.exists():
    C.freeze_prereg()
    print('PREREG frozen now (before any exp_4 fit)')
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp4_protocol as P, exp4_aux as A, exp2_protocol as P2, exp3_protocol as P3, exp3_touch as S

PREREG frozen now (before any exp_4 fit)
PREREG verified 254e2e216cb05154 | frozen 2026-11-09T17:09:15+00:00


## 1. Unit tests (synthetic arrays)
Bit-exact equivalence with `exp1_net.train` (aux off, no touch), aux targets from training matches only, mirror rules, `fit=False` refusal, job locks.

In [2]:
import subprocess
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'tests'],
                        capture_output=True, text=True, cwd=ROOT)
print(result.stdout.strip().splitlines()[-1])
assert result.returncode == 0, 'unit tests failed'

16 passed, 1 warning in 3.11s


## 2. Inputs read in place
exp_2 caches (fingerprint-checked), exp_3 touch-feature caches, and the provider training targets: the cached tokens' own state / boundary must equal `exp3_protocol.step_labels(·, 'provider')` on every training array.

In [3]:
for arm, fp in P.EXP2_FINGERPRINTS.items():
    missing = [k for k, *_ in P.P1.job_keys() if not (P2.prediction_folder(arm) / f'{k}.npz').exists()]
    print(f'exp_2 {arm:12s} fingerprint {fp} | missing jobs {len(missing)}')
    assert not missing and P2.fingerprint(arm) == fp
jobs = [(n, k, r) for n in C.MATCH_IDS for r, k in P.RECIPE]
assert all(S.feature_path(n, k, r).exists() for n, k, r in jobs + [(n, 'target_like', 0) for n in C.MATCH_IDS])
same = []
for name, kind, round_index in jobs:
    tokens = P2.load_tokens(name, kind, round_index)
    state, boundary = P3.step_labels(name, 'provider', tokens)
    same.append(bool(np.array_equal(state, tokens['state']) and np.array_equal(boundary, tokens['boundary'])))
print('touch caches present for', len(jobs) + len(C.MATCH_IDS), 'arrays | provider targets equal cached targets:', all(same), f'({len(same)} arrays)')
assert all(same)

exp_2 exp2_aug     fingerprint 5cea8203869e | missing jobs 0
exp_2 exp2_aug_s1  fingerprint 01776e3c0b77 | missing jobs 0
exp_2 exp2_aug_s2  fingerprint fb137265b072 | missing jobs 0


touch caches present for 28 arrays | provider targets equal cached targets: True (21 arrays)


## 3. Auxiliary targets (clean IDSSE data)
On every training array and both held-out views, where the observed ball is visible the aux ball target must equal the observed `ball_x` / `ball_y` globals up to float16 rounding (this checks the normalisation and the flip sign). Reported: share of hidden-ball steps that get a ball target, and how often the clean nearest team equals the provider possession team on alive steps (descriptive only).

In [4]:
rows = []
for name in C.MATCH_IDS:
    arrays = [(kind, r, P2.load_tokens(name, kind, r)) for r, kind in P.RECIPE]
    arrays += [(view, 0, P2.view_tokens(name, view)) for view in C.VIEWS]
    for kind, r, tokens in arrays:
        rows.append({'match': name, 'kind': kind, 'round': r, **A.consistency(tokens, name, r)})
    rows[-1]['team_agrees_with_provider_alive'] = A.team_agreement(name, P2.load_tokens(name)['state'])
aux = pd.DataFrame(rows)
aux.to_csv(C.RESULTS / 'aux_integrity.csv', index=False)
print(aux.groupby('kind')[['visible_steps', 'max_abs_diff_visible', 'hidden_steps', 'hidden_with_ball_target', 'team_target_defined', 'steps_sign_differs_from_clean']].agg(['mean', 'max']).round(4).to_string())
print('team agreement on alive steps (mean of 7):', round(aux.team_agrees_with_provider_alive.mean(), 4))
assert aux.max_abs_diff_visible.max() <= 2e-3 and (aux.visible_without_clean_ball == 0).all()

            visible_steps        max_abs_diff_visible         hidden_steps        hidden_with_ball_target      team_target_defined      steps_sign_differs_from_clean       
                     mean    max                 mean     max         mean    max                    mean  max                mean  max                          mean    max
kind                                                                                                                                                                        
original       16906.0000  17326               0.0005  0.0005   11741.7143  12037                     1.0  1.0                 1.0  1.0                        0.0000      0
target_aug     11284.0000  13104               0.0005  0.0005   17363.7143  18675                     1.0  1.0                 1.0  1.0                      989.8571  13858
target_like    11324.5714  13369               0.0005  0.0005   17323.1429  18318                     1.0  1.0                 1.0  1.0

## 4. Reference code path
`exp3_ref_ens3` is read from exp_3's stored nested results. One of them is recomputed here with the same code and must match exactly.

In [5]:
t0 = time.time()
fresh = P3.evaluate_predictions(P.predictions_for('ref_ens3', {}), 'exp3_ref_ens3', 'target_like', 'provider')
reused = P.reused_result('exp3_ref_ens3', 'target_like', 'provider')
check = {'rows_equal': fresh['rows'] == reused['rows'], 'final_config_equal': fresh['final_config'] == reused['final_config'],
         'mean_spell_f1': float(np.mean([r['spell_f1'] for r in fresh['rows']])), 'seconds': round(time.time() - t0, 1)}
C1.write_json(C.RESULTS / 'reference_recheck.json', check)
print(check)
assert check['rows_equal'] and check['final_config_equal']

{'rows_equal': True, 'final_config_equal': True, 'mean_spell_f1': 0.6641362312691413, 'seconds': 72.1}


In [6]:
print('exp_4 fits so far:', P.fit_count())
print(pd.DataFrame(P.spec_table()).to_string(index=False))

exp_4 fits so far: 0
          arm  ball_weight  team_weight  smooth_l1_beta  head_seed_offset  touch     seed  fingerprint
    exp4_full         0.25         0.25             1.0           9000000   True 20261108 32589590c3f9
 exp4_full_s1         0.25         0.25             1.0           9000000   True 20261109 d1f0c79a7ac4
 exp4_full_s2         0.25         0.25             1.0           9000000   True 20261110 764527653714
   exp4_touch         0.00         0.00             1.0           9000000   True 20261108 2d42a3706658
exp4_touch_s1         0.00         0.00             1.0           9000000   True 20261109 78a2fbc5df96
exp4_touch_s2         0.00         0.00             1.0           9000000   True 20261110 4329112f1f34
     exp4_aux         0.25         0.25             1.0           9000000  False 20261108 ffb5f8550d22
  exp4_aux_s1         0.25         0.25             1.0           9000000  False 20261109 2483e220c97f
  exp4_aux_s2         0.25         0.25             